In [1]:
import h5py
import numpy as np
import os
import sys
from shutil import copyfile

# Importa suas ferramentas
sys.path.insert(0, os.path.abspath(".."))
from tools import utils
from tqdm import tqdm

In [2]:
# ================= CONFIGURAÇÃO =================
SNAP_NUM = "264"
MATCH_INDEX = 250  
XMATCH_DIR = "../xmatchs"
# Caminho da simulação ORIGINAL
PATH_SIM_ORIGINAL = "/home/STORAGE/MN5_resims/fiducial/hydro_output"
SIM_NAME = "fiducial"

# Onde salvar a mini simulação?
OUTPUT_BASE_DIR = "./mini_fiducial_test" 

# Parâmetros do Recorte
R200_FACTOR = 100.0  # Raio do recorte = 10 * R200
# ================================================

In [3]:
def smart_read(dataset, mask):
    """Lê dataset com fallback para uint64 se for uint48"""
    try:
        data = dataset[:]
        return data[mask]
    except TypeError:
        shape = dataset.shape
        buffer = np.empty(shape, dtype=np.uint64)
        dataset.read_direct(buffer)
        return buffer[mask]

def create_mini_simulation():
    print(f"--> Inicializando leitura do Halo {MATCH_INDEX} na simulação {SIM_NAME}...")
    
    # 1. Dados do Halo Original
    matcher = utils.HaloMatcher(XMATCH_DIR)
    ctx = utils.SimulationContext(PATH_SIM_ORIGINAL, SNAP_NUM)
    halo_finder = utils.HaloFinder(ctx)

    halo_id = matcher.get_halo_id(SIM_NAME, MATCH_INDEX)
    halo_props = halo_finder.get_halo_properties(halo_id)
    
    original_box = halo_props['box_size']
    r200 = halo_props['r200']
    halo_center_original = halo_props['cm']
    
    cutout_radius = R200_FACTOR * r200
    new_box_size = (cutout_radius * 2.0) * 1.1 
    new_center = np.array([new_box_size/2, new_box_size/2, new_box_size/2])
    
    print(f"Halo ID: {halo_id} | R200: {r200:.4f} | Nova Box: {new_box_size:.4f}")

    snap_dir = os.path.join(OUTPUT_BASE_DIR, f"snapdir_{SNAP_NUM}")
    group_dir = os.path.join(OUTPUT_BASE_DIR, f"groups_{SNAP_NUM}")
    os.makedirs(snap_dir, exist_ok=True)
    os.makedirs(group_dir, exist_ok=True)

    # --- PARTE A: SNAPSHOT ---
    out_snap_file = os.path.join(snap_dir, f"snapshot_{SNAP_NUM}.0.hdf5")
    first_snap = ctx.get_snap_file(0)
    
    with h5py.File(first_snap, 'r') as f_in, h5py.File(out_snap_file, 'w') as f_out:
        # Header Básico
        header = f_out.create_group("Header")
        for k, v in f_in['Header'].attrs.items(): header.attrs[k] = v
        
        # Header Crítico
        header.attrs['BoxSize'] = new_box_size
        header.attrs['NumFilesPerSnapshot'] = 1
        
        # --- FIX DAS UNIDADES (CRUCIAL) ---
        # Tenta pegar de Parameters ou usa Default para simulação em Mpc
        units_found = False
        if "Parameters" in f_in:
            f_in.copy("Parameters", f_out)
            # Atualiza BoxSize nos parametros
            if "BoxSize" in f_out["Parameters"].attrs:
                f_out["Parameters"].attrs["BoxSize"] = new_box_size
                
            # Tenta copiar as Units de Parameters -> Header
            params = f_in["Parameters"].attrs
            unit_keys = ['UnitLength_in_cm', 'UnitMass_in_g', 'UnitVelocity_in_cm_per_s']
            
            for u in unit_keys:
                if u in params:
                    header.attrs[u] = params[u]
                    units_found = True
        
        # Se não achou nos Parameters, HARDCODE valores padrão para Arepo em Mpc
        if not units_found or 'UnitLength_in_cm' not in header.attrs:
            print("--> AVISO: Unidades não encontradas. Aplicando padrão Mpc/1e10Msun.")
            header.attrs['UnitLength_in_cm'] = 3.085678e24   # 1 Mpc
            header.attrs['UnitMass_in_g'] = 1.989e43         # 1e10 Msun
            header.attrs['UnitVelocity_in_cm_per_s'] = 100000.0 # 1 km/s

        n_files_orig = int(f_in['Header'].attrs['NumFilesPerSnapshot'])
        total_parts_by_type = np.zeros(6, dtype=np.int64)

        # LOOP PRINCIPAL
        for pt_idx in range(6):
            pt_name = f"PartType{pt_idx}"
            accumulated_data = {}
            has_data = False
            
            for i in tqdm(range(n_files_orig), desc=f"Processando {pt_name}", unit="arq"):
                try:
                    f_chunk = h5py.File(ctx.get_snap_file(i), 'r')
                except: continue
                    
                if pt_name not in f_chunk: f_chunk.close(); continue

                coords = f_chunk[pt_name]['Coordinates'][:]
                dx = coords - halo_center_original
                dx[dx > original_box/2] -= original_box
                dx[dx < -original_box/2] += original_box
                
                dist_sq = np.sum(dx**2, axis=1)
                mask = dist_sq <= cutout_radius**2
                
                if np.sum(mask) > 0:
                    has_data = True
                    new_coords = dx[mask] + new_center
                    for key in f_chunk[pt_name].keys():
                        if key not in accumulated_data: accumulated_data[key] = []
                        if key == 'Coordinates': 
                            accumulated_data[key].append(new_coords)
                        else: 
                            dset = f_chunk[pt_name][key]
                            accumulated_data[key].append(smart_read(dset, mask))
                f_chunk.close()
            
            if has_data:
                g_out = f_out.create_group(pt_name)
                for key, data_list in accumulated_data.items():
                    full_data = np.concatenate(data_list, axis=0)
                    g_out.create_dataset(key, data=full_data, compression="gzip")
                    if key == 'Coordinates': total_parts_by_type[pt_idx] = full_data.shape[0]

        header.attrs['NumPart_ThisFile'] = total_parts_by_type
        header.attrs['NumPart_Total'] = total_parts_by_type
        header.attrs['NumPart_Total_HighWord'] = np.zeros(6, dtype=np.int64)

    # --- PARTE B: GRUPO ---
    print("--> Gerando Catálogo de Grupo...")
    out_group_file = os.path.join(group_dir, f"fof_subhalo_tab_{SNAP_NUM}.0.hdf5")
    with h5py.File(ctx.get_group_file(0), 'r') as g_in, h5py.File(out_group_file, 'w') as g_out:
        header = g_out.create_group("Header")
        for k, v in g_in['Header'].attrs.items(): header.attrs[k] = v
        header.attrs['BoxSize'] = new_box_size
        header.attrs['Ngroups_Total'] = 1; header.attrs['Ngroups_ThisFile'] = 1
        header.attrs['NumFiles'] = 1
        if "Parameters" in g_in: g_in.copy("Parameters", g_out)
        
        grp = g_out.create_group("Group")
        grp.create_dataset("GroupPos", data=np.array([new_center]))
        grp.create_dataset("Group_M_Crit200", data=np.array([halo_props['mass']]))
        grp.create_dataset("Group_R_Crit200", data=np.array([r200]))
        scale_factor = header.attrs['Time']
        raw_vel = halo_props['vel_pec'] / np.sqrt(scale_factor)
        grp.create_dataset("GroupVel", data=np.array([raw_vel]))

    print(f"\n--> SUCESSO! Salvo em: {OUTPUT_BASE_DIR}")

In [4]:
if __name__ == "__main__":
    create_mini_simulation()

2026-02-10 15:00:46,007 - INFO - Loading match catalog for fiducial...


--> Inicializando leitura do Halo 250 na simulação fiducial...
Halo ID: 928 | R200: 0.1188 | Nova Box: 26.1405


Processando PartType5: 100%|██████████| 8/8 [00:00<00:00, 960.65arq/s]

--> Gerando Catálogo de Grupo...

--> SUCESSO! Salvo em: ./mini_fiducial_test
